In [12]:
import os

import numpy as np

os.chdir("..")

In [16]:
from biosym.model import model

In [17]:
m = model.load_model("tests/test_models/gait2d_torque/gait2d_torque.yaml")

Loading model from cache: b8c190b2ad6ff9e048b91ca3b5ccf3257478a584270b4f3e5734fe368d073af9.cpkl


In [21]:
m.ext_torques

{'names': ['t_foot_l_x',
  't_foot_l_y',
  't_foot_l_z',
  't_foot_r_x',
  't_foot_r_y',
  't_foot_r_z'],
 'idx': 39,
 'n': 6}

In [4]:
import jax
import jax.numpy as jnp
from jax.experimental.ode import odeint

# Initial state as a dict
y0 = {
    "position": jnp.array([1.0, 0.0]),
    "velocity": jnp.array([0.0, 1.0]),
}

# Time points
t = jnp.array([0.0, 0.1])


# ODE function that returns a dict of the same structure
def dynamics(y, t, constants):
    pos = y["position"]
    vel = y["velocity"]
    acceleration = -constants["k"] * pos  # e.g., spring
    return {"position": vel, "velocity": acceleration}


# Constants also as a dict
constants = {"k": 1.0}

# Integrate
result = odeint(jax.jit(dynamics), y0, t, constants)

# result is a PyTree of shape (len(t), ...) matching y0
print(result["position"][1])
print(result["velocity"][1])

[0.9950043  0.09983344]
[-0.09983344  0.9950043 ]


In [10]:
force = np.array([1, 0])
cop = np.array([0, -1])
np.cross(cop, force)

/var/folders/zp/1cs5lqq978n6gd_m465gfm840000gn/T/ipykernel_11051/543519310.py:4: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  np.cross(cop, force)


array(1)

In [3]:
import jax
def jaxfun(dict):
    return dict['a'] + dict['b']

In [5]:
d = {'a': 1, 'b': 2}
print(jax.jit(jaxfun)(d))

3


In [14]:
d_long = [d] * 10
print(jax.vmap(jax.jit(jaxfun), 0,d_long)(d_long))

ValueError: vmap was requested to map its argument along axis 0, which implies that its rank should be at least 1, but is only 0 (its shape is ())

In [63]:
import numpy as np
dict = {'a': {'aa': np.random.random((2, 3)), 'ab': np.ones((2, 3))},
        'b': {'ba': np.zeros((2, 3)), 'bb': np.ones((2, 3))}}

In [64]:
def get_single_state(states_dict, n):
    """
    Get a single state from the states dictionary.
    :param states_dict: The states dictionary.
    :param n: The index of the state to retrieve.
    :return: The state at index n.
    """
    # Loop over the nested dictionary (depth=2) and return the state at index n
    return {key: {k: v[n] for k, v in value.items()} for key, value in dict.items()}
print(get_single_state(dict, 0))

{'a': {'aa': array([0.22971396, 0.76184485, 0.65256781]), 'ab': array([1., 1., 1.])}, 'b': {'ba': array([0., 0., 0.]), 'bb': array([1., 1., 1.])}}


In [65]:
states_list = [get_single_state(dict, i) for i in range(2)]

In [66]:
def states_list_to_dict(states_list):
    """
    Convert a list of states to a dictionary format.
    :param states_list: List of states.
    :return: Dictionary with states as keys and their values as lists.
    """
    return {key: {k: np.vstack([state[key][k] for state in states_list]) for k, v in value.items()} for key, value in states_list[0].items()}

In [67]:
states_list_to_dict(states_list)

{'a': {'aa': array([[0.22971396, 0.76184485, 0.65256781],
         [0.52090312, 0.07990513, 0.71613052]]),
  'ab': array([[1., 1., 1.],
         [1., 1., 1.]])},
 'b': {'ba': array([[0., 0., 0.],
         [0., 0., 0.]]),
  'bb': array([[1., 1., 1.],
         [1., 1., 1.]])}}

In [80]:
def sum_states_dicts(states_list, weights=None):
    """
    Sum a list of states dictionaries, optionally applying weights.
    :param states_list: List of states dictionaries.
    :param weights: Optional list of weights to apply to each state.
    :return: A single states dictionary with summed values.
    """
    if weights is None:
        weights = [1] * len(states_list)
    
    return {key: {k: np.sum(np.array([state[key][k] * weight for state, weight in zip(states_list, weights)]), axis=0)
                 for k, v in value.items()} for key, value in states_list[0].items() if key != 'input_names'}


In [81]:
sum_states_dicts([dict, dict])

{'a': {'aa': array([[0.45942792, 1.5236897 , 1.30513562],
         [1.04180625, 0.15981025, 1.43226104]]),
  'ab': array([[2., 2., 2.],
         [2., 2., 2.]])},
 'b': {'ba': array([[0., 0., 0.],
         [0., 0., 0.]]),
  'bb': array([[2., 2., 2.],
         [2., 2., 2.]])}}

In [73]:
a[[1,2,2], [0, 1, 2]]  # This will raise an error because the indices are out of bounds

array([ 4,  9, 10])

In [74]:
a

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

In [82]:
from collections import namedtuple

# Define a named tuple for the states
States = namedtuple('States', ['position', 'velocity'])


In [86]:
States.position = np.array([1.0, 0.0])

In [88]:
States['position']

__main__.States['position']

In [94]:
def create_named_tuple_from_states(states_dict):
    """
    Create a named tuple from a states dictionary.
    :param states_dict: Dictionary of states.
    :return: Named tuple with states as fields.
    """
    fields = list(states_dict.keys())
    States = namedtuple('States', fields)
    for key, value in states_dict.items():
        States[key] = namedtuple(list(states_dict[key].keys()))
    return States


In [95]:
create_named_tuple_from_states(dict)

TypeError: namedtuple() missing 1 required positional argument: 'field_names'

In [96]:
def dict_to_namedtuple(name, d):
    """Convert a dictionary `d` into a namedtuple called `name`."""
    def convert(key, value):
        if isinstance(value, dict):
            return dict_to_namedtuple(key.capitalize(), value)
        elif isinstance(value, list):
            return [convert(f"{key}_{i}", item) for i, item in enumerate(value)]
        else:
            return value

    fields = {key: convert(key, value) for key, value in d.items()}
    return namedtuple(name, fields.keys())(**fields)

In [98]:
from flax import struct
from dataclasses import fields, is_dataclass
import jax.numpy as jnp

def get_state_row(obj, idx):
    """
    Recursively extract the idx-th row from all JAX arrays in a flax.struct.dataclass.
    Preserves the dataclass structure.
    
    :param obj: A flax.struct.dataclass or nested structure.
    :param idx: Index to extract (along axis 0).
    :return: A new structure with only the idx-th row from each array field.
    """
    if is_dataclass(obj):
        return obj.__class__(**{
            f.name: get_state_row(getattr(obj, f.name), idx)
            for f in fields(obj)
        })
    elif isinstance(obj, jnp.ndarray) and obj.ndim > 0:
        return obj[idx]
    else:
        return obj  # Pass through scalars or non-array fields



ModuleNotFoundError: No module named 'flax'

NameError: name 'struct' is not defined

In [ ]:
def n_ways_to_stairs_climb_to_get_to_the_top(n_stairs):
    """
        This fn.
    """
    assert 1 <= n_stairs <= 45, f"Invalid number of stairs ({n_stairs}), must be between 1,45"

    res = 1

    def helper(n_stairs_left):
        nonlocal res
        if n_stairs_left < 2: 
            return
        res += 1

        helper(n_stairs_left-1)
        helper(n_stairs_left-2)
    
    helper(n_stairs)
    return res



In [128]:
n_ways_to_stairs_climb_to_get_to_the_top(6)

13

In [ ]:
1: 1
2: 2
3: 3
4: 5
5: 8 
6: 13

In [140]:
def dynamic_arrays(n):
    arr_i_0 = np.zeros(n)

    arr_i_0[1] = 1
    arr_i_0[2] = 2

    for i in range(3,n):
        arr_i_0[i] = arr_i_0[i-1] + arr_i_0[i-2]

    return arr_i_0

In [142]:
dynamic_arrays(45)[-1]

np.float64(1134903170.0)

In [145]:
def fib(n):
    n = n+1
    return 1/np.sqrt(5)* ((1 + np.sqrt(5)) / 2)**n - (1/np.sqrt(5))*((1 - np.sqrt(5)) / 2)**n

In [192]:
def maxprof(arr):
    max_prof = 0
    last_prof = 0
    last_prof_old = 0
    i_buy = 0
    i_sell = 0
    i_buy_curr = 0

    for i in range(1, len(arr)):
        last_prof += arr[i] - arr[i-1]
        if last_prof < 0:
            last_prof = 0
            i_buy_curr = i
        if last_prof < last_prof_old:
            print(i)
            if max_prof < last_prof_old:
                i_buy = i_buy_curr
                i_sell = i - 1
                max_prof = last_prof_old
        last_prof_old = last_prof
    if last_prof > max_prof:
        i_sell = i
        max_prof = last_prof   
    return (i_buy, i_sell, max_prof)
            
                


In [201]:
a = [7,6,4,3,1]
b = [7,1,5,3,6,4]
c = [0,10,10,-1,10]

In [202]:
maxprof(a), maxprof(b), maxprof(c)

3
5
3


((0, 0, 0), (1, 4, 5), (3, 4, 11))

In [18]:
from flax.struct import dataclass, field
import flax.struct as fs
import jax.numpy as jnp 

@dataclass
class SubDataClass:
    model: jnp.ndarray
    gc_model: jnp.ndarray
    actuator_model: jnp.ndarray = None

@dataclass
class MyDataClass:
    states: SubDataClass
    constants: SubDataClass
    input_names: tuple = field(default=("model",))

    def at(self, idx):
        """
        Get a single state from the states dictionary.
        :param idx: The index of the state to retrieve.
        :return: The state at index idx.
        """
        return MyDataClass(
            states=SubDataClass(
                model=self.states.model[idx],
                gc_model=self.states.gc_model[idx] if self.states.gc_model.size > 0 else jnp.zeros(0),
                actuator_model=self.states.actuator_model[idx] if self.states.actuator_model.size > 0 else jnp.zeros(0),
            ),
            constants=SubDataClass(
                model=self.constants.model[idx],
                gc_model=self.constants.gc_model[idx] if self.constants.gc_model.size > 0 else jnp.zeros(0),
                actuator_model=self.constants.actuator_model[idx] if self.constants.actuator_model.size > 0 else jnp.zeros(0),
            ),
            input_names=self.input_names
        )





In [19]:
zeros = lambda size: jnp.zeros(size)
default = MyDataClass(
    states=SubDataClass( model=zeros((12,10)),
                    gc_model=zeros(0),
                    actuator_model=zeros(0)),
    constants=SubDataClass( model=zeros((12,10)),
                       gc_model=zeros(0)),
    input_names=("model",))

In [15]:
default.at(0)

MyDataClass(states=SubDataClass(model=Array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32), gc_model=Array([], shape=(0,), dtype=float32), actuator_model=Array([], shape=(0,), dtype=float32)), constants=SubDataClass(model=Array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32), gc_model=Array([], shape=(0,), dtype=float32), actuator_model=Array([], shape=(0,), dtype=float32)), input_names=('model',))

In [ ]:
default

MyDataClass(states=SubDataClass(model=Array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32), gc_model=Array([], shape=(0,), dtype=float32), actuator_model=Array([], shape=(0,), dtype=float32)), constants=SubDataClass(model=Array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.,

In [ ]:
def aaaa(a,v,c):
    return (a,v,c)

